# GEOS curves as true curves in ArcGIS Pro

Demo of `geos_true_curves.py`. **This is a reconstruction, not the original
`GEOS_Test_notebook.ipynb`, and it has no GEOS logo gear cells** (that file was not available when this
was written):
it keeps the same idea, loading `geos_c.dll` with ctypes, building curves from WKT and
mapping them, but every cell is new. Curve parameters that were not known are marked *assumed*.

Run it inside ArcGIS Pro (Windows). `GEOSCurveToLine` + `arcpy.FromWKT` gives straight pieces;
`GEOSEsriJSONWriter` + `arcpy.AsShape(json, True)` gives circular arcs, elliptic arcs and
cubic Beziers.

In [ ]:
import ctypes, math, os, sys
import arcpy

HERE = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, HERE)
import geos_true_curves as gtc

# Path to the geos_c.dll built from the true-curves branch (next to geos.dll).
GEOS_DLL = os.path.join(HERE, "geos_c.dll")
geos = ctypes.CDLL(GEOS_DLL)

geos.initGEOS.argtypes = [ctypes.c_void_p, ctypes.c_void_p]
geos.initGEOS(None, None)
geos.GEOSGeomFromWKT.argtypes = [ctypes.c_char_p]
geos.GEOSGeomFromWKT.restype = ctypes.c_void_p
geos.GEOSGeom_destroy.argtypes = [ctypes.c_void_p]
geos.GEOSCurveToLine.argtypes = [ctypes.c_void_p, ctypes.c_void_p]
geos.GEOSCurveToLine.restype = ctypes.c_void_p
geos.GEOSCurveToLineParams_create.restype = ctypes.c_void_p
geos.GEOSCurveToLineParams_destroy.argtypes = [ctypes.c_void_p]
geos.GEOSCurveToLineParams_setMaxDeviation.argtypes = [ctypes.c_void_p, ctypes.c_double]
geos.GEOSGeomToWKT.argtypes = [ctypes.c_void_p]
geos.GEOSGeomToWKT.restype = ctypes.c_void_p
geos.GEOSFree.argtypes = [ctypes.c_void_p]
gtc.bind(geos)   # the Esri JSON function

In [ ]:
def from_wkt(wkt):
    g = geos.GEOSGeomFromWKT(wkt.encode())
    if not g:
        raise ValueError("GEOS could not parse: " + wkt[:80])
    return g

def to_arcpy_linearized(geom, max_deviation=0.01):
    # The old path, kept for comparison: GEOSCurveToLine, then FromWKT.
    params = geos.GEOSCurveToLineParams_create()
    geos.GEOSCurveToLineParams_setMaxDeviation(params, max_deviation)
    line = geos.GEOSCurveToLine(geom, params)
    geos.GEOSCurveToLineParams_destroy(params)
    ptr = geos.GEOSGeomToWKT(line)
    wkt = ctypes.string_at(ptr).decode()
    geos.GEOSFree(ptr)
    geos.GEOSGeom_destroy(line)
    return arcpy.FromWKT(wkt)

def to_arcpy_true_curve(geom, tolerance=0.01):
    # Exact where GEOS has an exact curve, Beziers within `tolerance` otherwise.
    return gtc.to_arcpy_true_curve(geos, geom, tolerance, with_stats=True)

def show(name, wkt, tolerance=0.01, true_curves=True):
    g = from_wkt(wkt)
    if true_curves:
        shape, stats = to_arcpy_true_curve(g, tolerance)
        print(name, stats)
    else:
        shape = to_arcpy_linearized(g)
        print(name, "linearized, %d vertices" % shape.pointCount)
    geos.GEOSGeom_destroy(g)
    return shape

In [ ]:
PLACE = "REFERENCELOCATION AFFINEPLACEMENT (LOCATION (%r %r), REFERENCEDIRECTIONS (VECTOR (%r %r), VECTOR (%r %r)))"
def placement(x=0.0, y=0.0, heading=0.0):
    c, s = math.cos(heading), math.sin(heading)
    return PLACE % (x, y, c, s, -s, c)

# Clothoid, 25 m at SCALEFACTOR 50.
clothoid = show("clothoid", "CLOTHOID (%s, SCALEFACTOR 50, STARTDISTANCE 0, ENDDISTANCE 25)" % placement(), 0.001)

# NURBS circle (rational quadratic, 9 control points): exact circular arcs.
w = math.sqrt(0.5)
def cp(x, y, wt): return "NURBSPOINT (WEIGHTEDPOINT (%r %r), WEIGHT %r)" % (wt * x, wt * y, wt)
circle = ("NURBSCURVE (DEGREE 2, CONTROLPOINTS (" + ", ".join([
    cp(1,0,1), cp(1,1,w), cp(0,1,1), cp(-1,1,w), cp(-1,0,1), cp(-1,-1,w), cp(0,-1,1), cp(1,-1,w), cp(1,0,1)])
    + "), KNOTS (KNOT (VALUE 0, MULTIPLICITY 3), KNOT (VALUE 0.25, MULTIPLICITY 2), KNOT (VALUE 0.5, MULTIPLICITY 2), "
    "KNOT (VALUE 0.75, MULTIPLICITY 2), KNOT (VALUE 1, MULTIPLICITY 3)))")
nurbs_circle = show("NURBS circle", circle, 0.001)

In [ ]:
# Spiral fan (assumed: five types, 1200 m, R = 400 m at the end, 15 degrees apart).
fan = []
for i, kind in enumerate(["clothoid", "bloss", "biquadratic", "sine", "cosine"]):
    wkt = ("SPIRALCURVE (%s, LENGTH 1200, STARTCURVATURE 0, ENDCURVATURE %r, SPIRALTYPE %s)"
           % (placement(0, 0, math.radians(15.0 * i)), 1 / 400.0, kind))
    fan.append(show("spiral " + kind, wkt, 0.01))

In [ ]:
# Railway alignment: see measure_true_curves.railway() for the CompoundCurve with bloss
# transitions (assumed: 400 m straights, 150 m transitions, R = 800 m).
import measure_true_curves as mtc
railway_wkt = mtc.railway(geos)
railway = show("railway", railway_wkt, 0.01)
railway_old = show("railway", railway_wkt, true_curves=False)

In [ ]:
# Put the shapes on a map: copy them to the in-memory workspace and add them.
shapes = {"clothoid": clothoid, "nurbs_circle": nurbs_circle, "railway": railway, "railway_linearized": railway_old}
shapes.update({"fan_%d" % i: s for i, s in enumerate(fan)})
aprx = arcpy.mp.ArcGISProject("CURRENT")
for name, shape in shapes.items():
    out = arcpy.management.CopyFeatures([shape], "memory\\" + name)
    aprx.activeMap.addDataFromPath(out.getOutput(0))